# GPT-2 124M — DIMER E2E domain-adaptation tutorial: perplexity on paper abstracts (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/gpt2-text-generation-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/gpt2-text-generation-pipeline/blob/main/tutorials/gpt2_text_generation_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-openai--community%2Fgpt2-ffcc4d?style=flat)](https://huggingface.co/openai-community/gpt2) [![Upstream](https://img.shields.io/badge/Upstream-openai%2Fgpt--2-181717?style=flat&logo=github&logoColor=white)](https://github.com/openai/gpt-2)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** causal text generation (continuing one English prompt, greedy by default, seeded nucleus sampling on request) and bounded causal-LM fine-tuning of the last transformer blocks on a text corpus, measured by held-out perplexity, using the pinned `openai-community/gpt2` weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/gpt2_text_generation_pipeline/`, at revision `ab9e64185bda`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `607a30d783dfa663caf39e06633721c8d4cfcd7e` (~554 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned GPT-2 snapshot (safetensors, 548 MB), fetches the three digest-pinned SciTLDR-A files from the project repository (5.5 MB, no credential), reads the paper abstracts and draws 300 / 50 / 100 training, validation and test documents from the release's own paper-disjoint members, generates greedy and seeded-sampled continuations of an unseen abstract's opening through the inference contract with an input manifest and a rejection probe, scores the frozen model on the test abstracts by held-out perplexity beside the add-one unigram floor, runs a bounded fine-tuning of the last four transformer blocks with validation-perplexity epoch selection, scores the held-out split again, continues the same unseen abstracts with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path takes about five minutes of model time after the downloads; a CUDA runtime is used automatically when present.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own text corpus as a CSV (columns `id`, `text`), a JSON array or JSONL file of `{{id, text}}` records, or a plain `.txt` file in which every blank-line-separated paragraph is one document. It passes through the same validation, seeded text-disjoint split, unigram floor, frozen scoring, fine-tuning, held-out evaluation, generation, artifact export and reload-parity cells as the SciTLDR sample. The expected schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

At inference the byte-level BPE tokenizer turns the prompt into token ids (no special tokens are added), and the 12-block decoder-only Transformer predicts one next-token distribution over the 50,257-token vocabulary at a time, feeding each chosen token back until `max_new_tokens` is reached or the end-of-text token is produced. **Two decoding modes are demonstrated and must not be confused:** greedy decoding (`do_sample=False`, the pipeline default) takes the argmax at every step and is deterministic on a fixed device and dtype; nucleus sampling (`do_sample=True` with `temperature`, `top_p` and a mandatory `seed`) draws from the truncated distribution, reproducible only for the same seed on the same host. GPT-2 is a **base language model**: no chat template, no instruction following, no safety tuning, English web text of 2019 vintage. The carried pipeline module adds manifest verification, input validation and ceilings (prompts are rejected, never truncated), a settings validator that refuses unseeded sampling, the fixed pad/EOS handling and a fixed output contract. **A generation carries no score**; the number a language model *does* have is how surprised it is by text it did not write.

What this notebook adds to inference is **domain adaptation measured by that number**. The dataset is real: SciTLDR-A (Cachola et al., 2020; Apache-2.0) ships the abstracts of 3,229 computer-science papers as three digest-pinned JSON-Lines files fetched from the project repository at a pinned commit; only the abstract text is used, so every record is one document and no reference is needed. The carried `metrics.py` turns the model's own teacher-forced per-token negative log-likelihoods into **perplexity** and **bits per token** on the held-out abstracts, and fits an **add-one unigram model** on the training tokens as the floor a model that ignores context reaches. The fine-tuning question is whether a bounded adaptation of the last transformer blocks on 300 abstracts lowers the perplexity of abstracts the model has not seen. Nothing here is a quality claim: a lower perplexity says the adapted model finds paper abstracts less surprising, not that it writes good ones.

**Learning objectives:** install the pinned runtime; read what the carried pipeline, dataset and metrics modules guarantee; stage and digest-verify the immutable upstream snapshot; fetch a digest-pinned referenced corpus and validate and split it without leakage; generate through the public API in both decoding modes and read `finished_by`, the token counts and the echoed settings correctly; read a perplexity beside a unigram floor and understand what it does and does not measure; run a bounded fine-tuning with explicit hyperparameters and validation-based epoch selection; evaluate on an independent test split; compare continuations before and after; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** chat or instruction following (GPT-2 has neither), batching (one prompt per call), raw logits or hidden states, beam search, non-English text, full-model or embedding fine-tuning, any quality, fluency or factuality score, and any claim that a SciTLDR abstract split stands in for your corpus. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available. CPU is adequate: the build record measured about 4 s to load and digest-verify the 548 MB snapshot, 8.5 s to score the 100-abstract test split (19,779 predicted tokens) and about 114 s per training epoch over 300 abstracts plus a 50-abstract validation pass per epoch. The pinned `torch==2.14.0` install and the 548 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python; what next-token prediction is; the difference between argmax decoding and sampling from a truncated distribution; what perplexity is (the exponential of the mean per-token negative log-likelihood) and why it is an intrinsic number, not a judgement of the text a model writes.
- **Data contract:** records are `{{id, text}}` — one document of the domain, 1..4,000 characters and at most 1,023 BPE tokens (a longer record is refused, not truncated, when scored), ids matching `[A-Za-z0-9_.:-]{{1,64}}` and unique; a dataset needs 8..20,000 records; texts are de-duplicated case-insensitively before splitting so the same document never sits in two splits; during training only, documents are truncated to 512 tokens and the end-of-text token is appended. BYOD accepts CSV, JSON, JSONL or TXT in that shape.
- **Validation is structural, not semantic:** nothing checks that a record belongs to the domain you mean — an off-topic corpus is fine-tuned on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — an internal document collection is exactly that. The default path uploads nothing. A base language model can continue any prompt with false, biased or offensive text — read the output before reusing it.
- **External access (data):** besides the Hub, the default path fetches three pinned objects (`train.jsonl` 3,155,015 bytes, `dev.jsonl` 1,124,865 bytes, `test.jsonl` 1,204,107 bytes; SHA-256 `b222771d…` / `3191fa98…` / `fb42dd6c…`) from `raw.githubusercontent.com` at the pinned `allenai/scitldr` commit over HTTPS, each refused on any mismatch before it is read; SciTLDR is Apache-2.0 (Cachola et al., 2020).
- **External access:** the Hugging Face Hub only, to fetch the pinned `openai-community/gpt2` snapshot (~554 MB in total) at revision `607a30d783df…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'tokenizers==0.22.2',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'gpt2-text-generation-pipeline',
    'repository_revision': 'ab9e64185bda29238d59713e9211ed2334db3bc2',
    'embedded_module': 'src/gpt2_text_generation_pipeline/pipeline.py',
    'embedded_modules': ['src/gpt2_text_generation_pipeline/metrics.py', 'src/gpt2_text_generation_pipeline/pipeline.py', 'src/gpt2_text_generation_pipeline/samples.py'],
    'module_sha256': '1cd894da305072518845e76f2c87ae13132acda1fbc3d4bc63c52ab9249d23b0',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/gpt2_text_generation_pipeline/` @ `ab9e64185bda`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/gpt2_text_generation_pipeline/metrics.py`

In [ ]:
"""Intrinsic language-model metrics over a held-out text corpus and the unigram baseline.

The model's own per-token negative log-likelihoods (teacher-forced, natural log) are aggregated into
**perplexity** (`exp(mean NLL)`) and **bits per token** (`mean NLL / ln 2`); the mean is taken over every
predicted token of every record (token-weighted), so long records count more than short ones. Perplexity
needs no references, only text: it says how surprised the model is by the domain, not whether its
generations are good, fluent or true. The **unigram baseline** is the perplexity of an add-one-smoothed
unigram model fitted on the training tokens — the floor a model that ignores context reaches.
"""

from __future__ import annotations

import math
from collections import Counter
from collections.abc import Sequence
from typing import Any

METRIC_DEFINITIONS = {
    "perplexity": (
        "exp of the token-weighted mean negative log-likelihood (natural log) of every predicted token in "
        "the held-out records under teacher forcing; lower is better; a record's first token is not predicted"
    ),
    "bits_per_token": "the same mean negative log-likelihood divided by ln 2",
    "mean_nll": "the token-weighted mean negative log-likelihood in nats",
}


def sequence_metrics(losses: Sequence[Sequence[float]]) -> dict[str, Any]:
    """Aggregate per-record lists of per-token NLLs (nats) into perplexity and bits per token."""
    if not losses:
        raise ValueError("no records to score")
    total = 0.0
    n_tokens = 0
    per_record = []
    for record_losses in losses:
        if not record_losses:
            raise ValueError("a record scored no tokens (it needs at least two)")
        s = float(sum(record_losses))
        total += s
        n_tokens += len(record_losses)
        per_record.append(math.exp(s / len(record_losses)))
    mean_nll = total / n_tokens
    return {
        "n_records": len(losses),
        "n_tokens": n_tokens,
        "mean_nll": mean_nll,
        "perplexity": math.exp(mean_nll),
        "bits_per_token": mean_nll / math.log(2),
        "record_perplexity": {
            "min": min(per_record),
            "median": sorted(per_record)[len(per_record) // 2],
            "max": max(per_record),
        },
        "definitions": dict(METRIC_DEFINITIONS),
    }


def unigram_losses(
    train_ids: Sequence[Sequence[int]], test_ids: Sequence[Sequence[int]], vocab_size: int
) -> list[list[float]]:
    """Per-token NLLs of an add-one-smoothed unigram model fitted on `train_ids`, scored on `test_ids`
    (skipping each record's first token, like the model's teacher-forced scoring)."""
    if vocab_size < 1:
        raise ValueError("vocab_size must be positive")
    counts: Counter[int] = Counter()
    for ids in train_ids:
        counts.update(int(t) for t in ids)
    total = sum(counts.values()) + vocab_size
    log_denominator = math.log(total)
    out = []
    for ids in test_ids:
        out.append([log_denominator - math.log(counts.get(int(t), 0) + 1) for t in ids[1:]])
    return out


def unigram_baseline(
    train_ids: Sequence[Sequence[int]], test_ids: Sequence[Sequence[int]], vocab_size: int
) -> dict[str, Any]:
    """Perplexity of the context-free unigram model — the floor a language model must beat."""
    result = sequence_metrics(unigram_losses(train_ids, test_ids, vocab_size))
    result["baseline"] = "add-one-smoothed unigram model fitted on the training tokens (ignores context)"
    return result

**Module 2/3:** `src/gpt2_text_generation_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""Causal text generation over the pinned ``openai-community/gpt2`` (124M) checkpoint.

Weights load only from a digest-verified local snapshot (``weights/<key>/``) or, when explicitly allowed,
from the Hugging Face Hub at the pinned revision. One task method, ``generate``: greedy decoding by default
(deterministic), nucleus sampling only when asked for and seeded. One prompt per call.

The adaptation contract (``evaluate``, ``unigram_baseline``, ``adapt``, ``save_artifact``, ``from_artifact``)
scores a validated ``{id, text}`` corpus by teacher-forced perplexity, fine-tunes the last transformer blocks
on it with validation-perplexity epoch selection, and exports the trained tensors as a safetensors adapter
bound to the pinned base weights. The inference contract above is unchanged by it.
"""

from __future__ import annotations

import hashlib
import json
import math
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

MODEL_ID = "openai-community/gpt2"
MODEL_REVISION = "607a30d783dfa663caf39e06633721c8d4cfcd7e"
MODEL_LICENSE = "mit"
MODEL_KEY = "gpt2"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Ceilings. CONTEXT_LENGTH is n_positions / n_ctx in the pinned config.json; prompt tokens plus new tokens
# must fit in it, so a prompt is rejected (never truncated) above MAX_PROMPT_TOKENS and the combined length
# is checked before the model runs. MAX_NEW_TOKENS bounds one call's cost on CPU.
CONTEXT_LENGTH = 1024
MAX_PROMPT_TOKENS = CONTEXT_LENGTH - 1  # leaves room for at least one generated token
MAX_NEW_TOKENS = 256
DEFAULT_MAX_NEW_TOKENS = 32
MAX_TEXT_CHARS = 4_000  # pre-tokenisation guard; ~4 chars per byte-level BPE token on English text
VOCAB_SIZE = 50257  # config.json vocab_size
EOS_TOKEN_ID = 50256  # config.json eos_token_id == bos_token_id; GPT-2 has no pad token, so pad = eos
PAD_TOKEN_ID = EOS_TOKEN_ID
DECODING_DEFAULT = "greedy"  # do_sample=False -> argmax over the next-token distribution at every step
WEIGHT_FILE = "model.safetensors"
WEIGHT_SHA256 = (
    "248dfc3911869ec493c76e65bf2fcf7f615828b0254c12b473182f0f81d3a707"  # manifest digest of WEIGHT_FILE
)
PARAMETER_COUNT = 124_439_808
TRANSFORMER_BLOCKS = 12  # config.json n_layer
DEFAULT_TRAINABLE_BLOCKS = 4  # the last four transformer blocks (28,351,488 parameters)
MAX_TRAIN_TOKENS = 512  # text truncation ceiling during adaptation (scoring never truncates — it rejects)
MAX_EVAL_RECORDS = 2_000
MAX_RECORDS_FIT = 20_000  # the unigram baseline may be fitted on a whole training split
MIN_SCORED_RECORDS = 50  # below this a scored corpus is labelled a small sample
ARTIFACT_FORMAT = "org.valcorza.gpt2.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def validate_settings(
    max_new_tokens: Any, do_sample: Any, temperature: Any, top_p: Any, seed: Any
) -> dict[str, Any]:
    """Check decoding settings before the model runs; sampling must be explicit and seeded."""
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int):
        raise TypeError("max_new_tokens must be an int")
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError(f"max_new_tokens must be between 1 and MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
    if not isinstance(do_sample, bool):
        raise TypeError("do_sample must be a bool")
    if isinstance(temperature, bool) or not isinstance(temperature, int | float) or not temperature > 0:
        raise ValueError("temperature must be a number > 0")
    if isinstance(top_p, bool) or not isinstance(top_p, int | float) or not 0 < top_p <= 1:
        raise ValueError("top_p must be a number in (0, 1]")
    if seed is not None and (isinstance(seed, bool) or not isinstance(seed, int) or seed < 0):
        raise TypeError("seed must be a non-negative int or None")
    if do_sample and seed is None:
        raise ValueError("seed is required when do_sample=True so that sampled output is reproducible")
    return {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "temperature": float(temperature) if do_sample else None,
        "top_p": float(top_p) if do_sample else None,
        "seed": seed if do_sample else None,
        "decoding": "nucleus-sampling" if do_sample else DECODING_DEFAULT,
    }


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one non-empty str prompt; no special tokens are added and the prompt is continued verbatim",
    "prompt_chars": [1, MAX_TEXT_CHARS],
    "prompt_tokens": [1, MAX_PROMPT_TOKENS],
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "context_length": CONTEXT_LENGTH,
    "temperature": "number > 0 (sampling only)",
    "top_p": "number in (0, 1] (sampling only)",
    "seed": "non-negative int, required when do_sample=True",
    "vocab_size": VOCAB_SIZE,
    "eos_token_id": EOS_TOKEN_ID,
    "pad_token_id": PAD_TOKEN_ID,
    "preprocessing": (
        "byte-level BPE with no special tokens; a prompt over MAX_PROMPT_TOKENS, or a prompt whose "
        "tokens plus max_new_tokens exceed CONTEXT_LENGTH, is rejected rather than truncated"
    ),
}


def _check_prompt(prompt: Any) -> str:
    """Raise TypeError/ValueError naming the first violated prompt ceiling; return the prompt."""
    if not isinstance(prompt, str):
        raise TypeError(f"prompt must be a str, got {type(prompt).__name__}")
    if not prompt.strip():
        raise ValueError("prompt must not be empty or whitespace only")
    if len(prompt) > MAX_TEXT_CHARS:
        raise ValueError(f"prompt has {len(prompt)} chars > MAX_TEXT_CHARS={MAX_TEXT_CHARS}")
    return prompt


def validate_inputs(
    prompt: str,
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    do_sample: bool = False,
    temperature: float = 1.0,
    top_p: float = 1.0,
    seed: int | None = None,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    Rejection is reported by raising exactly as ``generate`` would: both route the prompt through
    ``_check_prompt`` and the decoding settings through the public ``validate_settings``. The two
    token ceilings (``MAX_PROMPT_TOKENS`` and ``prompt + max_new_tokens <= CONTEXT_LENGTH``) need
    the loaded tokenizer and are therefore enforced inside ``generate``, not here.
    """
    checked = _check_prompt(prompt)
    settings = validate_settings(max_new_tokens, do_sample, temperature, top_p, seed)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry: generate takes one prompt per call")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "prompt-0", "chars": len(checked)}],
        "settings": settings,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], references: Sequence[str] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even though no metric exists here.

    A free-text continuation has no ground truth and the repository ships no metric helper, so the
    verdict is always ``not-measurable`` (EVAL9). ``references`` exists for interface parity with
    the fleet's other pipelines and is recorded in ``reason`` rather than scored: perplexity needs a
    held-out corpus scored by the model, not a reference string compared to one completion, and any
    quality judgement needs human raters or a labelled downstream task.
    """
    settings = result.get("settings", {})
    supplied = references is not None
    return {
        "task": "causal text generation (continuing one prompt)",
        "score_semantics": (
            "the completion carries no score, probability or confidence; `finished_by` says whether "
            "the end-of-text token or the token budget stopped it, and the echoed `settings` say how "
            f"it was decoded ({settings.get('decoding', DECODING_DEFAULT)})"
        ),
        "sample_kind": sample_kind,
        "n_new_tokens": int(result.get("new_tokens", 0)),
        "metrics": [],
        "baselines": [],
        "verdict": "not-measurable",
        "reason": (
            "a continuation has no ground truth and the repository ships no metric helper"
            + (
                "; references were supplied but no metric helper exists to score them here, and a "
                "reference string is not a corpus"
                if supplied
                else "; the evaluated sample has no reference corpus"
            )
        ),
        "needs": (
            "a held-out reference corpus from the deployment domain, scored for perplexity with the "
            "caller's own code, for an intrinsic number; or human raters, or a labelled downstream "
            "task, for any quality or factuality claim — none of which this repository ships"
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


@dataclass
class GPT2TextGenerationPipeline:
    """``_tokenize`` maps text to token ids, ``_runner`` maps (prompt ids, settings) to new token ids, and
    ``_decode`` maps ids back to text; all three are injectable so tests run offline."""

    _tokenize: Callable[[str], list[int]]
    _runner: Callable[[list[int], dict[str, Any]], list[int]]
    _decode: Callable[[list[int]], str]
    device: str = "cpu"
    source: str = "injected"
    _scorer: Callable[[list[int]], list[float]] | None = field(default=None, repr=False)
    adapter: dict[str, Any] | None = field(default=None, repr=False)
    _model: Any = field(default=None, repr=False)
    _tokenizer: Any = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> GPT2TextGenerationPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
        elif allow_download:
            source, kwargs = MODEL_ID, {}
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import GPT2LMHeadModel, GPT2TokenizerFast

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        tokenizer = GPT2TokenizerFast.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = GPT2LMHeadModel.from_pretrained(
            source, revision=MODEL_REVISION, dtype=torch.float32, trust_remote_code=False, **kwargs
        )
        model = model.to(resolved_device).eval()

        def runner(prompt_ids: list[int], settings: dict[str, Any]) -> list[int]:
            input_ids = torch.tensor([prompt_ids], dtype=torch.long, device=resolved_device)
            gen_kwargs: dict[str, Any] = {
                "max_new_tokens": settings["max_new_tokens"],
                "do_sample": settings["do_sample"],
                "pad_token_id": PAD_TOKEN_ID,
                "eos_token_id": EOS_TOKEN_ID,
            }
            if settings["do_sample"]:
                gen_kwargs.update(temperature=settings["temperature"], top_p=settings["top_p"])
                torch.manual_seed(settings["seed"])
            with torch.inference_mode():
                output = model.generate(input_ids, attention_mask=torch.ones_like(input_ids), **gen_kwargs)
            return output[0, len(prompt_ids) :].tolist()

        def tokenize(text: str) -> list[int]:
            return tokenizer(text, add_special_tokens=False)["input_ids"]

        def scorer(ids: list[int]) -> list[float]:
            """Per-token NLLs (nats) of ids[1:] given the preceding tokens, under teacher forcing."""
            input_ids = torch.tensor([ids], dtype=torch.long, device=resolved_device)
            with torch.inference_mode():
                logits = model(input_ids=input_ids, attention_mask=torch.ones_like(input_ids)).logits
            log_probs = torch.log_softmax(logits[0, :-1].float(), dim=-1)
            return (-log_probs.gather(1, input_ids[0, 1:, None])[:, 0]).tolist()

        source_kind = "local-snapshot" if kwargs else "hf-hub"
        return cls(
            tokenize,
            runner,
            tokenizer.decode,
            resolved_device,
            source_kind,
            _scorer=scorer,
            _model=model,
            _tokenizer=tokenizer,
        )

    def generate(
        self,
        prompt: str,
        *,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
        do_sample: bool = False,
        temperature: float = 1.0,
        top_p: float = 1.0,
        seed: int | None = None,
    ) -> dict[str, Any]:
        """Continue one prompt. Greedy (deterministic) unless ``do_sample=True`` with a ``seed``."""
        prompt = _check_prompt(prompt)
        settings = validate_settings(max_new_tokens, do_sample, temperature, top_p, seed)
        prompt_ids = list(self._tokenize(prompt))
        if not 1 <= len(prompt_ids) <= MAX_PROMPT_TOKENS:
            raise ValueError(
                f"prompt has {len(prompt_ids)} tokens, outside 1..MAX_PROMPT_TOKENS={MAX_PROMPT_TOKENS}"
            )
        if len(prompt_ids) + settings["max_new_tokens"] > CONTEXT_LENGTH:
            raise ValueError(
                f"prompt tokens {len(prompt_ids)} + max_new_tokens {settings['max_new_tokens']} "
                f"> CONTEXT_LENGTH={CONTEXT_LENGTH}"
            )
        new_ids = [int(t) for t in self._runner(prompt_ids, settings)]
        if len(new_ids) > settings["max_new_tokens"] or any(not 0 <= t < VOCAB_SIZE for t in new_ids):
            raise RuntimeError("runner returned more than max_new_tokens tokens, or an id outside the vocab")
        finished_by = "eos" if EOS_TOKEN_ID in new_ids else "max_new_tokens"
        kept = new_ids[: new_ids.index(EOS_TOKEN_ID)] if finished_by == "eos" else new_ids
        completion = self._decode(kept) if kept else ""
        return {
            "prompt": prompt,
            "completion": completion,
            "text": prompt + completion,
            "prompt_tokens": len(prompt_ids),
            "new_tokens": len(kept),
            "finished_by": finished_by,
            "settings": settings,
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- adaptation -----------------------------------------------------------------------------------

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._tokenizer is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return self._model, self._tokenizer

    def _record_ids(self, records: Sequence[Mapping[str, Any]]) -> list[list[int]]:
        """Tokenise validated records; a record is refused (never truncated) above MAX_PROMPT_TOKENS or
        below two tokens (one token predicts nothing)."""
        out = []
        for record in records:
            ids = list(self._tokenize(record["text"]))
            if len(ids) > MAX_PROMPT_TOKENS:
                raise ValueError(
                    f"record {record['id']} has {len(ids)} tokens; ceiling is {MAX_PROMPT_TOKENS}"
                )
            if len(ids) < 2:
                raise ValueError(f"record {record['id']} tokenises to fewer than two tokens")
            out.append(ids)
        return out

    def evaluate(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Score every record by teacher-forced perplexity (natural-log NLL per predicted token)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import sequence_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if self._scorer is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        losses = [self._scorer(ids) for ids in self._record_ids(checked)]
        metrics = sequence_metrics(losses)
        metrics.update(
            {
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    def unigram_baseline(
        self, train: Sequence[Mapping[str, Any]], test: Sequence[Mapping[str, Any]]
    ) -> dict[str, Any]:
        """Perplexity of an add-one unigram model over the GPT-2 vocabulary, fitted on `train`, on `test`."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import unigram_baseline` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        train_checked = validate_dataset(train, min_records=1, max_records=MAX_RECORDS_FIT)["records"]
        test_checked = validate_dataset(test, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        return unigram_baseline(self._record_ids(train_checked), self._record_ids(test_checked), VOCAB_SIZE)

    def _trainable_names(self, trainable_blocks: int) -> list[str]:
        if not isinstance(trainable_blocks, int) or not 1 <= trainable_blocks <= TRANSFORMER_BLOCKS:
            raise ValueError(f"trainable_blocks must be an int in 1..{TRANSFORMER_BLOCKS}")
        model, _ = self._require_model()
        first = TRANSFORMER_BLOCKS - trainable_blocks
        prefixes = tuple(f"transformer.h.{k}." for k in range(first, TRANSFORMER_BLOCKS))
        return [name for name, _p in model.named_parameters() if name.startswith(prefixes)]

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 2,
        lr: float = 1e-4,
        batch_size: int = 8,
        trainable_blocks: int = DEFAULT_TRAINABLE_BLOCKS,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded causal-LM fine-tuning on a validated text corpus.

        Only the last `trainable_blocks` transformer blocks train (4 by default; the token and position
        embeddings, the tied output projection, the final layer norm and the earlier blocks stay frozen).
        Next-token cross-entropy on every token of every record (the end-of-text token is appended so the
        model also learns where a document ends), AdamW at a fixed learning rate with gradient clipping at
        1.0, records truncated to MAX_TRAIN_TOKENS **during training only**. Epoch 0 records the frozen
        model's validation perplexity; the epoch with the lowest validation perplexity is kept."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not (0.0 < lr <= 1e-3):
            raise ValueError("lr must be in (0, 1e-3]")
        if not isinstance(batch_size, int) or not 1 <= batch_size <= 32:
            raise ValueError("batch_size must be an int in 1..32")
        names = self._trainable_names(trainable_blocks)
        train_checked = validate_dataset(train)["records"]
        val_checked = (
            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"] if val else []
        )
        train_ids = [ids[:MAX_TRAIN_TOKENS] + [EOS_TOKEN_ID] for ids in self._record_ids(train_checked)]
        import torch

        torch.manual_seed(seed)
        model, _ = self._require_model()
        started = time.perf_counter()
        wanted = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
        device = torch.device(self.device)

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            return {
                k: v
                for k, v in self.evaluate(val_checked).items()
                if k in ("perplexity", "bits_per_token", "n_tokens")
            }

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}
        history.append(entry)
        if progress:
            progress(entry)
        best_ppl = entry["val"]["perplexity"] if entry["val"] else math.inf
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        best_epoch = 0
        generator = torch.Generator().manual_seed(seed)
        for epoch in range(1, epochs + 1):
            model.train()
            order = torch.randperm(len(train_ids), generator=generator).tolist()
            losses = []
            for start in range(0, len(order), batch_size):
                batch = [train_ids[i] for i in order[start : start + batch_size]]
                width = max(len(ids) for ids in batch)
                input_ids = torch.full((len(batch), width), PAD_TOKEN_ID, dtype=torch.long)
                attention = torch.zeros((len(batch), width), dtype=torch.long)
                labels = torch.full((len(batch), width), -100, dtype=torch.long)
                for row, ids in enumerate(batch):
                    input_ids[row, : len(ids)] = torch.tensor(ids)
                    attention[row, : len(ids)] = 1
                    labels[row, : len(ids)] = torch.tensor(ids)
                out = model(
                    input_ids=input_ids.to(device),
                    attention_mask=attention.to(device),
                    labels=labels.to(device),
                )
                optimiser.zero_grad(set_to_none=True)
                out.loss.backward()
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                optimiser.step()
                losses.append(float(out.loss.detach()))
            model.eval()
            entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": score_val()}
            history.append(entry)
            if progress:
                progress(entry)
            current = entry["val"]["perplexity"] if entry["val"] else -math.inf
            if current < best_ppl or not entry["val"]:
                best_ppl = current
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
                best_epoch = epoch
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable_blocks": trainable_blocks,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": "lowest validation perplexity"
            if val_checked
            else "final epoch (no validation split)",
            "lr": lr,
            "batch_size": batch_size,
            "max_train_tokens": MAX_TRAIN_TOKENS,
            "n_train": len(train_checked),
            "n_train_tokens": sum(len(ids) for ids in train_ids),
            "n_val": len(val_checked),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts ------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted transformer-block tensors as safetensors with a manifest naming the base."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        model, _ = self._require_model()
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "weight_file": WEIGHT_FILE,
                "weight_sha256": WEIGHT_SHA256,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest and digest, then overwrite exactly the tensors it carries."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
            MODEL_ID,
            MODEL_REVISION,
            WEIGHT_SHA256,
        ):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        entry = manifest["files"][0]
        weights_path = root / entry["path"]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        model, _ = self._require_model()
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != manifest["tensors"]:
            raise ValueError("artifact tensor names differ from its manifest")
        state = model.state_dict()
        for key, value in tensors.items():
            if key not in state or not key.startswith("transformer.h."):
                raise ValueError(
                    f"artifact tensor {key} is not an adaptable transformer-block tensor of the base"
                )
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(
                    f"artifact tensor {key}: shape {tuple(value.shape)} != {tuple(state[key].shape)}"
                )
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})
        model.load_state_dict(merged, strict=True)
        model.eval()
        self.adapter = {
            **manifest["adapter"],
            "trainable_names": manifest["tensors"],
            "history": manifest.get("history", []),
        }
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> GPT2TextGenerationPipeline:
        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 3/3:** `src/gpt2_text_generation_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Text-corpus dataset contract for domain adaptation of the language model: the pinned SciTLDR sample of
paper abstracts, validation, seeded splitting, BYOD loaders and CSV export.

The default corpus is **real** and far from GPT-2's WebText pre-training distribution: the abstracts of
computer-science papers shipped by SciTLDR (Cachola et al., EMNLP Findings 2020; Apache-2.0). The three
`SciTLDR-A` JSON-Lines files are fetched one by one from the project repository at a pinned commit and refused
on any byte-size or SHA-256 mismatch; only the abstract text is used here. The frozen model's held-out
perplexity on abstracts is the number to beat, and the fine-tuning question is whether a bounded adaptation
of the last transformer blocks lowers it on abstracts the model has not seen.

A record is ``{id, text}``: one document of the domain, no prompt and no reference — perplexity needs none.
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import re
import urllib.request
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_TEXT_CHARS, MODEL_ID` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "SciTLDR-A (paper abstracts)"
CORPUS_RELEASE = "allenai/scitldr @ 5ccad9c00a60ad75c9e04abf7f27d0f53f983b20"
CORPUS_BASE_URL = "https://raw.githubusercontent.com/allenai/scitldr/5ccad9c00a60ad75c9e04abf7f27d0f53f983b20/SciTLDR-Data/SciTLDR-A/"
CORPUS_FILES = {
    "train": ("train.jsonl", 3_155_015, "b222771d387be585cfdf5ae957b36757138415a352e0a3e3b23f73f87c3b1119"),
    "dev": ("dev.jsonl", 1_124_865, "3191fa98ccc09521332b7a1cd63b1930be4e8df125a235ccd31e40329709525e"),
    "test": ("test.jsonl", 1_204_107, "fb42dd6cd4f4a1928ae8a01a189456fbfe994a07e938bd49f68653933f6503c9"),
}
CORPUS_LICENSE = "Apache-2.0 (Cachola et al. 2020; allenai/scitldr)"
CORPUS_PAPERS = {"train": 1_992, "dev": 619, "test": 618}
DEFAULT_CACHE_DIR = Path("weights") / "scitldr"
MAX_SAMPLE_TEXT_CHARS = 2_400  # longer abstracts are left out of the sample (the ceiling is 1,023 tokens)
MIN_SAMPLE_TEXT_CHARS = 200
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 300, "validation": 50, "test": 100}
MIN_RECORDS = 8
MAX_RECORDS = 20_000
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, bytes]:
    """Return the three pinned SciTLDR-A files (bytes) from the cache or the project repository, verified."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    out = {}
    for split, (name, size, digest) in CORPUS_FILES.items():
        local = cache / name
        data = local.read_bytes() if local.is_file() else b""
        if len(data) != size or _sha256_bytes(data) != digest:
            url = CORPUS_BASE_URL + name
            if fetcher is not None:
                data = fetcher(url)
            else:
                with urllib.request.urlopen(url, timeout=120) as response:  # noqa: S310 (pinned https URL)
                    data = response.read()
            if len(data) != size or _sha256_bytes(data) != digest:
                raise ValueError(
                    f"{name}: fetched {len(data)} bytes with sha256 {_sha256_bytes(data)[:16]}…, "
                    f"pinned {size} / {digest[:16]}…"
                )
            local.write_bytes(data)
        out[split] = data
    return out


def read_corpus(files: Mapping[str, bytes]) -> dict[str, list[dict[str, Any]]]:
    """Parse the JSON-Lines members into abstract records keeping each SciTLDR `paper_id`."""
    out = {}
    for split in CORPUS_FILES:
        if split not in files:
            raise ValueError(f"corpus is missing the {split} file")
        records = []
        for line in files[split].decode("utf-8").splitlines():
            if not line.strip():
                continue
            row = json.loads(line)
            records.append(
                {
                    "id": f"{split}-{row['paper_id']}",
                    "text": " ".join(str(s).strip() for s in row["source"]),
                    "paper_id": str(row["paper_id"]),
                }
            )
        if len(records) != CORPUS_PAPERS[split]:
            raise ValueError(f"{split}: {len(records)} papers, expected {CORPUS_PAPERS[split]}")
        out[split] = records
    return out


def filter_records(records: Sequence[Mapping[str, Any]]) -> list[dict[str, Any]]:
    """Keep records whose text is within the sample length window; drop repeated texts case-insensitively."""
    seen: set[str] = set()
    kept = []
    for record in records:
        text = str(record["text"])
        if not MIN_SAMPLE_TEXT_CHARS <= len(text) <= MAX_SAMPLE_TEXT_CHARS:
            continue
        key = text.lower()
        if key in seen:
            continue
        seen.add(key)
        kept.append(dict(record))
    return kept


def build_sample_dataset(
    corpus: Mapping[str, Sequence[Mapping[str, Any]]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded draws from the three SciTLDR members: training from `train`, validation from `dev`, test from
    `test` — the release's own paper-disjoint partition, re-checked on texts by `check_split_disjoint`."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    source_of = {"train": "train", "validation": "dev", "test": "test"}
    rng = random.Random(seed)
    out: dict[str, list[dict[str, Any]]] = {}
    for name, size in sizes.items():
        pool = filter_records(corpus[source_of[name]])
        if size > len(pool):
            raise ValueError(f"requested {size} {name} records but only {len(pool)} fit")
        rng.shuffle(pool)
        out[name] = [
            {"id": f"{name}-{i:04d}", "text": r["text"], "paper_id": r["paper_id"]}
            for i, r in enumerate(pool[:size])
        ]
    return out


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Any = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus."""
    return build_sample_dataset(
        read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes
    )


def _check_record(record: Any, index: int) -> dict[str, Any]:
    label = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label} must be a mapping with id/text")
    for key in ("id", "text"):
        if key not in record:
            raise ValueError(f"{label} is missing {key!r}")
    rid, text = record["id"], record["text"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label}: id must match {_ID_RE.pattern}")
    if not isinstance(text, str):
        raise ValueError(f"{label}: text must be a string")
    if not text.strip():
        raise ValueError(f"{label}: text is empty")
    if len(text) > MAX_TEXT_CHARS:
        raise ValueError(f"{label}: text has {len(text)} chars; ceiling is MAX_TEXT_CHARS={MAX_TEXT_CHARS}")
    item = {"id": rid, "text": text.strip()}
    if "paper_id" in record:
        item["paper_id"] = str(record["paper_id"])
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a text corpus; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):
        raise ValueError("records must be a list of {id, text} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    texts: set[str] = set()
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        texts.add(item["text"].lower())
        checked.append(item)
    return {
        "records": checked,
        "n_records": len(checked),
        "unique_texts": len(texts),
        "text_chars": {
            "min": min(len(r["text"]) for r in checked),
            "max": max(len(r["text"]) for r in checked),
        },
        "total_chars": sum(len(r["text"]) for r in checked),
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [[r["id"], r["text"]] for r in records]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no lower-cased text appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = str(record["text"]).lower()
            if key in seen and seen[key] != name:
                raise ValueError(f"a text ({record['text'][:60]!r}…) appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD corpus into train/validation/test after de-duplicating texts."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    unique = []
    for record in checked:
        key = record["text"].lower()
        if key not in seen:
            seen.add(key)
            unique.append(record)
    random.Random(seed).shuffle(unique)
    n_test = max(1, round(len(unique) * test_fraction))
    n_val = round(len(unique) * val_fraction)
    splits = {
        "test": unique[:n_test],
        "validation": unique[n_test : n_test + n_val],
        "train": unique[n_test + n_val :],
    }
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(
            f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required"
        )
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read `{id, text}` records from a CSV (columns id, text), a JSON array or JSONL of such objects, or a
    plain `.txt` file in which every non-empty line (or blank-line-separated paragraph) is one record."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"dataset not found: {file_path}")
    suffix = file_path.suffix.lower()
    text = file_path.read_text(encoding="utf-8")
    if suffix == ".csv":
        rows = list(csv.DictReader(io.StringIO(text)))
        missing = {"id", "text"} - set(rows[0].keys() if rows else set())
        if missing:
            raise ValueError(f"CSV is missing columns {sorted(missing)}")
        return [{"id": r["id"], "text": r["text"]} for r in rows]
    if suffix == ".jsonl":
        return [json.loads(line) for line in text.splitlines() if line.strip()]
    if suffix == ".json":
        data = json.loads(text)
        if not isinstance(data, list):
            raise ValueError("JSON dataset must be an array of records")
        return data
    if suffix == ".txt":
        paragraphs = [p.strip() for p in text.replace("\r\n", "\n").split("\n\n") if p.strip()]
        return [{"id": f"doc-{i:05d}", "text": " ".join(p.split())} for i, p in enumerate(paragraphs)]
    raise ValueError("BYOD corpora must be .csv, .json, .jsonl or .txt")


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["id", "text"])
        writer.writeheader()
        for record in records:
            writer.writerow({"id": record["id"], "text": record["text"]})
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `15`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `607a30d783df…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `GPT2TextGenerationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "gpt2",
  "modelId": "openai-community/gpt2",
  "revision": "607a30d783dfa663caf39e06633721c8d4cfcd7e",
  "files": [
    {
      "path": "README.md",
      "bytes": 8092,
      "sha256": "0fcd631078093c2aa1d93438b898320b8a1167784e2a1ab37b8016e9de8b3c2e"
    },
    {
      "path": "config.json",
      "bytes": 665,
      "sha256": "0daed7749b4f02b8f76240d5444551d7b08712dab4d0adb8239c56ba823bb7b4"
    },
    {
      "path": "generation_config.json",
      "bytes": 124,
      "sha256": "ed0b32ac72c0f5f44a719abb2d7786ea5146c871f83717b7f2018065954de02b"
    },
    {
      "path": "merges.txt",
      "bytes": 456318,
      "sha256": "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5"
    },
    {
      "path": "model.safetensors",
      "bytes": 548105171,
      "sha256": "248dfc3911869ec493c76e65bf2fcf7f615828b0254c12b473182f0f81d3a707"
    },
    {
      "path": "onnx/config.json",
      "bytes": 879,
      "sha256": "c6d8a78631f7a03a14493eb78d584ba92c8e68c8774b810426b34df1f8a15b10"
    },
    {
      "path": "onnx/generation_config.json",
      "bytes": 119,
      "sha256": "067a873d1d1a67ffa7237a0ad0eebdb547d3f9209793c8aa476ec130815e3c8c"
    },
    {
      "path": "onnx/merges.txt",
      "bytes": 456318,
      "sha256": "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5"
    },
    {
      "path": "onnx/special_tokens_map.json",
      "bytes": 99,
      "sha256": "6f50ab5a5a509a1c309d6171f339b196a900dc9c99ad0408ff23bb615fdae7ad"
    },
    {
      "path": "onnx/tokenizer.json",
      "bytes": 2107653,
      "sha256": "cda20b8ca044949aa07ac4078420c80d1a57139d5f9f33700e46fb2d891e7c66"
    },
    {
      "path": "onnx/tokenizer_config.json",
      "bytes": 234,
      "sha256": "551e26ec611d8d0c8edc3ef72e518a38418cb71f40de1347dd486a595e1557d7"
    },
    {
      "path": "onnx/vocab.json",
      "bytes": 798156,
      "sha256": "3ba3c3109ff33976c4bd966589c11ee14fcaa1f4c9e5e154c2ed7f99d80709e7"
    },
    {
      "path": "tokenizer.json",
      "bytes": 1355256,
      "sha256": "8414cab924d8b9b33013f0d221c5862f365ee9be39c5c2bfae8a5a9e970478a6"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 26,
      "sha256": "5e04eb606e3a1583530a42e36c2a6b6615c86f34fe77e44d9ddeb43ff940931f"
    },
    {
      "path": "vocab.json",
      "bytes": 1042301,
      "sha256": "196139668be63f3b5d6574427317ae82f612a97c5d1cdaf36ed2256dbf636783"
    }
  ],
  "totalBytes": 554331411
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = GPT2TextGenerationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Referenced corpus, validation and split

`fetch_corpus` downloads the three pinned SciTLDR-A files (or reads them from the cache), refuses a byte-size or SHA-256 mismatch per file before it is parsed, and `read_corpus` flattens each JSON-Lines member into records whose `text` is the abstract's sentences joined by a space — titles and TLDRs are left unread. `build_sample_dataset` keeps abstracts of 200..2,400 characters, drops repeated texts, and draws 300 training documents from the `train` member, 50 validation documents from `dev` and 100 test documents from `test` by a seeded shuffle — the release's own paper-disjoint partition. `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no text appears in two splits, and the training split is written to `outputs/gpt2_text_generation_train.csv` in the shape BYOD expects.

Look for: 1,992 + 619 + 618 raw papers, three digests, splits 300 / 50 / 100, and four refusal probes — a duplicate id, an empty text, a missing field and a dataset too small to split — each rejected before `torch` does anything.

In [ ]:
import hashlib
import io
import json

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_papers = {'byod': len(records)}
else:
    corpus = read_corpus(fetch_corpus(cache_dir='weights/scitldr'))
    raw_papers = {name: len(part) for name, part in corpus.items()}
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} ({CORPUS_RELEASE}; {CORPUS_LICENSE})'
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
disjoint = check_split_disjoint(splits)
write_dataset_csv(train_records, 'outputs/gpt2_text_generation_train.csv')
print({'data_source': data_source, 'raw_papers': raw_papers, 'splits': disjoint, 'file_sha256': {k: v[2][:12] + '...' for k, v in CORPUS_FILES.items()}})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'unique_texts': manifest['unique_texts'], 'text_chars': manifest['text_chars'], 'total_chars': manifest['total_chars'], 'digest': manifest['digest'][:16] + '...'}})
print({'example': {'id': train_records[0]['id'], 'text': train_records[0]['text'][:200] + '...'}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'empty text': [{**train_records[0], 'text': '   '}, *train_records[1:8]],
    'missing field': [{'id': r['id']} for r in train_records[:8]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Generate through the inference contract in both decoding modes

Before any adaptation, the inference contract is exercised as it always was. The prompt is the opening of one test abstract (about 120 characters); `validate_inputs` applies exactly the checks `generate` applies (text type and character ceiling, `max_new_tokens` and the sampling settings within their ceilings) and returns an input manifest; the prompt-token ceilings need the real tokenizer and are enforced inside `generate`, which **rejects, never truncates**. An unseeded sampling request is validated too and its rejection recorded as a finding. `generate` returns `completion`, `text`, `prompt_tokens`, `new_tokens`, `finished_by` (`eos` when the model emitted token 50256 — GPT-2 ships no pad token, so the pipeline fixes `pad_token_id = eos_token_id = 50256` — or `max_new_tokens`) and echoes the settings. The greedy call is made twice and must repeat byte-identically (the reproducibility contract, on one device and dtype); the seeded sample is made twice and must repeat for the same seed. **Score semantics:** the pipeline emits **no probability, confidence or score of any kind** with a generation. Three further unseen-abstract openings are continued greedily here and kept as the *before* column for Section 9.

In [ ]:
import time

GREEDY_MAX_NEW_TOKENS = 32  # @param {type:"integer"}
SAMPLE_MAX_NEW_TOKENS = 32  # @param {type:"integer"}
TEMPERATURE = 0.8  # @param {type:"number"}
TOP_P = 0.9  # @param {type:"number"}
SEED = 7  # @param {type:"integer"}

def opening(record, chars=120):
    head = record['text'][:chars]
    return head.rsplit(' ', 1)[0] if ' ' in head else head

prompt = opening(test_records[0])
ceilings = {'CONTEXT_LENGTH': CONTEXT_LENGTH, 'MAX_PROMPT_TOKENS': MAX_PROMPT_TOKENS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'VOCAB_SIZE': VOCAB_SIZE, 'EOS_TOKEN_ID': EOS_TOKEN_ID, 'PAD_TOKEN_ID': PAD_TOKEN_ID, 'MAX_TRAIN_TOKENS': MAX_TRAIN_TOKENS}
print(ceilings)
input_manifest = validate_inputs(prompt, max_new_tokens=GREEDY_MAX_NEW_TOKENS, names=['test-opening'])
sampling_settings = validate_settings(SAMPLE_MAX_NEW_TOKENS, True, TEMPERATURE, TOP_P, SEED)
input_manifest['sampling_settings'] = sampling_settings
try:
    validate_inputs(prompt, max_new_tokens=SAMPLE_MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE, top_p=TOP_P)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'unseeded-sampling-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/gpt2_text_generation_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
started = time.perf_counter()
greedy = pipe.generate(prompt, max_new_tokens=GREEDY_MAX_NEW_TOKENS)
greedy_seconds = round(time.perf_counter() - started, 3)
greedy_repeat = pipe.generate(prompt, max_new_tokens=GREEDY_MAX_NEW_TOKENS)
sampled = pipe.generate(prompt, max_new_tokens=SAMPLE_MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE, top_p=TOP_P, seed=SEED)
sampled_repeat = pipe.generate(prompt, max_new_tokens=SAMPLE_MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE, top_p=TOP_P, seed=SEED)
checks = {
    'greedy_settings_echoed': greedy['settings'] == input_manifest['settings'] and greedy['settings']['decoding'] == 'greedy',
    'sampled_settings_echoed': sampled['settings'] == sampling_settings and sampled['settings']['seed'] == SEED,
    'new_tokens_within_budget': greedy['new_tokens'] <= GREEDY_MAX_NEW_TOKENS and sampled['new_tokens'] <= SAMPLE_MAX_NEW_TOKENS,
    'prompt_tokens_within_ceiling': 1 <= greedy['prompt_tokens'] <= MAX_PROMPT_TOKENS,
    'text_is_prompt_plus_completion': greedy['text'] == greedy['prompt'] + greedy['completion'],
    'finished_by_is_known': greedy['finished_by'] in ('eos', 'max_new_tokens'),
    'greedy_repeat_is_identical': greedy_repeat['completion'] == greedy['completion'],
    'same_seed_reproduces': sampled_repeat['completion'] == sampled['completion'],
}
if not all(checks.values()):
    raise RuntimeError(f'generate output failed a sanity check: {checks}')
print({'prompt_tokens': greedy['prompt_tokens'], 'greedy': {'new_tokens': greedy['new_tokens'], 'finished_by': greedy['finished_by'], 'seconds': greedy_seconds}, 'sampled': {'new_tokens': sampled['new_tokens'], 'finished_by': sampled['finished_by'], 'decoding': sampled['settings']['decoding']}, 'checks': checks, 'findings': len(input_manifest['findings']), 'no_score': 'the pipeline emits no probability or quality score'})
print(f'prompt:  {prompt!r}')
print(f"greedy:  {greedy['completion']!r}")
print(f"sampled: {sampled['completion']!r}")
if USE_BYOD:
    unseen_records = [{**r, 'id': f'unseen-{i:02d}'} for i, r in enumerate(test_records[1:4])]
else:
    used = {r['text'].lower() for part in splits.values() for r in part}
    unseen_records = [{**r, 'id': f'unseen-{i:02d}'} for i, r in enumerate([r for r in filter_records(corpus['dev']) if r['text'].lower() not in used][:3])]
before = {r['id']: pipe.generate(opening(r), max_new_tokens=GREEDY_MAX_NEW_TOKENS)['completion'] for r in unseen_records}
print({'unseen_openings_continued_by_the_frozen_model': len(before)})

## 6. The unigram floor and the frozen model's perplexity on the test split

Two numbers frame the adaptation. `pipe.unigram_baseline` fits an add-one-smoothed unigram model over the 50,257-token vocabulary on the training tokens and scores the test tokens with it: the perplexity a model that knows the domain's word frequencies but ignores every context reaches — expect a number in the thousands. `pipe.evaluate` scores the same test abstracts under teacher forcing with the frozen model: every token after a record's first is predicted from the tokens before it, the per-token negative log-likelihoods are averaged token-weighted and exponentiated (`perplexity`) or divided by ln 2 (`bits_per_token`); `record_perplexity` gives the spread across documents. Records over `MAX_PROMPT_TOKENS` are refused, never truncated. The build record saw the frozen model near 40 on these abstracts (WebText of 2019 already contains scientific prose); about ten seconds on CPU.

In [ ]:
t0 = time.perf_counter()
unigram = pipe.unigram_baseline(train_records, test_records)
print({'unigram_floor': {'perplexity': round(unigram['perplexity'], 1), 'bits_per_token': round(unigram['bits_per_token'], 3), 'n_tokens': unigram['n_tokens'], 'baseline': unigram['baseline']}, 'seconds': round(time.perf_counter() - t0, 1)})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records)
print({'frozen_model_test': {'perplexity': round(frozen_test['perplexity'], 2), 'bits_per_token': round(frozen_test['bits_per_token'], 3), 'n_records': frozen_test['n_records'], 'n_tokens': frozen_test['n_tokens'], 'record_perplexity': {k: round(v, 1) for k, v in frozen_test['record_perplexity'].items()}, 'verdict': frozen_test['verdict'], 'adapted': frozen_test['adapted']}, 'seconds': round(time.perf_counter() - t0, 1)})
print({'definitions': frozen_test['definitions']})
assert frozen_test['n_tokens'] == unigram['n_tokens'] and frozen_test['perplexity'] < unigram['perplexity']

## 7. Bounded fine-tuning on the training abstracts

`pipe.adapt` trains only the last `TRAINABLE_BLOCKS` transformer blocks — four by default, 28,351,488 of 124,439,808 parameters; the token and position embeddings, the tied output projection, the final layer norm and the earlier blocks stay frozen — with next-token cross-entropy on every token of every training abstract (the end-of-text token is appended so the model also learns where a document ends), AdamW at a fixed learning rate, gradient clipping at 1.0, seeded shuffling and no scheduler. Documents are truncated to `MAX_TRAIN_TOKENS` (512) **during training only**. Epoch 0 records the frozen model's validation perplexity; every epoch is scored the same way, and the epoch with the lowest validation perplexity is kept.

Watch validation perplexity fall from about 41 by a few points over two epochs (about 114 s of training plus a validation pass per epoch on CPU). The build record's sweep on this sample: two blocks at 5e-5 reached 37.39, two blocks at 1e-4 reached 36.32, four blocks at 1e-4 reached 34.73 in about the same time — the default.

In [ ]:
EPOCHS = 2  # @param {type:"integer"}
LEARNING_RATE = 1e-4  # @param {type:"number"}
BATCH_SIZE = 8  # @param {type:"integer"}
TRAINABLE_BLOCKS = 4  # @param {type:"integer"}

def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row['val_perplexity'] = round(entry['val']['perplexity'], 2)
        row['val_bits_per_token'] = round(entry['val']['bits_per_token'], 3)
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)

t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_blocks=TRAINABLE_BLOCKS, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'train_tokens': adapt_result['n_train_tokens'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test split was never used for training or epoch selection, and no abstract in it appears in the training or validation splits. The adapted model is scored exactly as the frozen model was in Section 6, and the three numbers are put side by side. Look for a perplexity a few points below the frozen one and far below the unigram floor; the cell asserts the adapted perplexity is lower than the frozen. One hundred abstracts from one seeded split of one corpus give no dispersion estimate; the delta is sample-sanity evidence that the adaptation contract works, not a benchmark, and a lower perplexity on paper abstracts says nothing about your corpus until you measure it there.

In [ ]:
adapted_test = pipe.evaluate(test_records)
adapted_val = pipe.evaluate(val_records)
comparison = {
    metric: {'unigram_floor': round(unigram[metric], 3), 'frozen': round(frozen_test[metric], 3), 'adapted': round(adapted_test[metric], 3)}
    for metric in ('perplexity', 'bits_per_token', 'mean_nll')
}
comparison['record_perplexity'] = {'frozen': {k: round(v, 1) for k, v in frozen_test['record_perplexity'].items()}, 'adapted': {k: round(v, 1) for k, v in adapted_test['record_perplexity'].items()}}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 3) for metric in ('perplexity', 'bits_per_token')}
for metric, row in comparison.items():
    print({metric: row})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'baselines': {'unigram_floor': unigram},
    'frozen_test': frozen_test,
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/gpt2_text_generation_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['perplexity'] < frozen_test['perplexity']
print({'report': 'outputs/gpt2_text_generation_evaluation_report.json'})

## 9. Continue unseen abstracts before and after, export the adapter and reload it

The three abstract openings continued by the frozen model in Section 5 are continued again by the adapted model through the same `generate` contract, and the two completions are printed side by side with the abstract's actual continuation. Read them as text, not as evidence: a perplexity gain is measured on the model's likelihoods, and nothing here scores a completion. The single-prompt `evaluation_report` helper — the inference-stage helper — is written for the first of them and stays `not-measurable`, because a continuation has no ground truth.

`pipe.save_artifact` writes the trained tensors — the last four transformer blocks, about 113 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `GPT2TextGenerationPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest and digest **before** deserialising, refuses any tensor that is not a transformer-block tensor of the base, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts an identical test perplexity on ten records and identical greedy completions (VER4).

In [ ]:
import csv
import shutil

rows = []
for record in unseen_records:
    head = opening(record)
    after = pipe.generate(head, max_new_tokens=GREEDY_MAX_NEW_TOKENS)
    rows.append({'id': record['id'], 'prompt': head, 'frozen_completion': before[record['id']], 'adapted_completion': after['completion'], 'actual_continuation': record['text'][len(head):len(head) + 160], 'prompt_tokens': after['prompt_tokens'], 'new_tokens': after['new_tokens'], 'finished_by': after['finished_by']})
    print({'id': record['id'], 'prompt': head[-60:]})
    print({'frozen': rows[-1]['frozen_completion']})
    print({'adapted': rows[-1]['adapted_completion']})
    print({'actual': rows[-1]['actual_continuation']})
single_report = evaluation_report(pipe.generate(opening(unseen_records[0]), max_new_tokens=GREEDY_MAX_NEW_TOKENS), sample_kind='one unseen SciTLDR abstract opening' if not USE_BYOD else 'one BYOD test record')
print({'single_prompt_report_verdict': single_report['verdict'], 'adapted_differs_from_frozen': sum(r['frozen_completion'] != r['adapted_completion'] for r in rows), 'of': len(rows)})
with open('outputs/gpt2_text_generation_completions.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

artifact_dir = Path('outputs/gpt2_text_generation_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'gpt2_text_generation', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = GPT2TextGenerationPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
parity_records = test_records[:10]
ppl_pair = (pipe.evaluate(parity_records)['perplexity'], reloaded.evaluate(parity_records)['perplexity'])
gen_pair = [(r['adapted_completion'], reloaded.generate(r['prompt'], max_new_tokens=GREEDY_MAX_NEW_TOKENS)['completion']) for r in rows]
parity = {'perplexity_in_memory': round(ppl_pair[0], 6), 'perplexity_reloaded': round(ppl_pair[1], 6), 'identical_completions': sum(a == b for a, b in gen_pair), 'of': len(gen_pair)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert abs(ppl_pair[0] - ppl_pair[1]) < 1e-6 and parity['identical_completions'] == parity['of']

weight_entry = next(entry for entry in snapshot['files'] if entry['path'] == WEIGHT_FILE)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHT_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': weight_entry['sha256']},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'release': CORPUS_RELEASE, 'base_url': CORPUS_BASE_URL, 'files': {k: {'name': v[0], 'bytes': v[1], 'sha256': v[2]} for k, v in CORPUS_FILES.items()}, 'license': CORPUS_LICENSE},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'prompt': prompt, 'greedy': {k: greedy[k] for k in ('completion', 'prompt_tokens', 'new_tokens', 'finished_by', 'settings')}, 'sampled': {k: sampled[k] for k in ('completion', 'prompt_tokens', 'new_tokens', 'finished_by', 'settings')}, 'seconds_greedy_first_call': greedy_seconds},
    'comparison': comparison,
    'completions_before_after': rows,
    'single_prompt_report': single_report,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device, 'dtype': 'float32', 'source': pipe.source},
}
with open('outputs/gpt2_text_generation_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The frozen 2019 model already finds paper abstracts far less surprising than a unigram model does (about 40 against a floor in the thousands — WebText contains scientific prose), and a bounded fine-tuning of the last four transformer blocks on 300 abstracts lowers held-out perplexity by a few points in a few minutes on CPU, with a 113 MB adapter that reloads to identical likelihoods and completions. That is the claim: the adaptation contract can adapt the model to a domain end to end on a real corpus, and the number it produces is read against the frozen model and a context-free floor rather than in isolation.

Perplexity is intrinsic: it says how well the model predicts text it did not write, token-weighted, on one seeded split of one corpus with no dispersion estimate. It is not fluency, factuality, usefulness or safety, and a lower perplexity does not make the completions in Section 9 better — they are unscored continuations from a base language model and can be false, repetitive, biased or offensive. The adapter changes the last blocks, which every prompt shares, so the model's behaviour on other text shifts too; nothing here measures that. Greedy decoding repeats itself; seeded sampling is reproducible only on the same host.

Three things to carry to real data. **Floors first:** the unigram floor and the frozen perplexity on *your* held-out documents are the numbers to read before any adapted one. **Leakage:** de-duplicate texts across splits (the contract does this case-insensitively) and split by document collection or author when your documents come from one. **Ceilings:** documents over `MAX_PROMPT_TOKENS` are refused when scored and truncated to 512 tokens only during training — long documents need chunking that this pipeline does not provide.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real corpus, validate the demonstrated dataset contract without leakage, execute the inference contract in both decoding modes and a bounded fine-tuning, evaluate by perplexity against a trivial floor and the frozen model on an independent split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, text quality or factual reliability on any domain, a usable acceptance threshold, or production fitness.

**Optional experiments (they do not affect the default path):** set `TRAINABLE_BLOCKS = 1` and watch the gain shrink; set `EPOCHS = 4` and watch whether validation perplexity keeps falling or turns (the best epoch is kept either way); change `SEED` in Section 5 for a different sampled continuation; or bring your own documents through BYOD and read the unigram floor before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/gpt2-text-generation-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/gpt2-text-generation-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/gpt2-text-generation-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/openai-community/gpt2
- Upstream code: https://github.com/openai/gpt-2
- Language Models are Unsupervised Multitask Learners (Radford et al., 2019; OpenAI technical report, no arXiv identifier): https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf
- TLDR: Extreme Summarization of Scientific Documents (Cachola et al., EMNLP Findings 2020; SciTLDR, Apache-2.0): https://arxiv.org/abs/2004.15011
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)